# 서브에이전트 위임

조사 역할을 서브에이전트로 분리할 때 전달되는 입력과 반환 결과를 확인합니다.

## 설치

고정된 패키지를 현재 커널에 설치합니다.

In [1]:
# 실습에서 검증한 패키지 버전을 현재 커널에 설치합니다.
%pip install -q deepagents==0.7.0 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

상위 폴더의 .env를 찾아 키 값은 출력하지 않고 버전만 확인합니다.

In [2]:
from importlib.metadata import version
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 현재 폴더부터 상위 폴더까지 .env를 찾고 값은 출력하지 않습니다.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
print('deepagents', version('deepagents'))
print('langchain', version('langchain'))
assert version('deepagents') == '0.7.0'

deepagents 0.7.0
langchain 1.4.0


## 단계 1

한 가지 동작을 실행하고 출력으로 확인합니다.

In [3]:
# 고정 자료 조회 도구와 검토 전용 서브에이전트를 준비합니다.
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from deepagents import create_deep_agent

@tool
def source_lookup(source_id: str) -> str:
    """Return one fixed, pre-verified course source."""
    return {'release':'DeepAgents 0.7 makes planning opt-in.'}[source_id]

researcher = {'name':'source-reviewer','description':'Check provided source claims.',
 'system_prompt':'Use source_lookup and return the source id and supported claim.', 'tools':[source_lookup]}
agent = create_deep_agent(model=init_chat_model('openai:gpt-5.4-mini'), tools=[source_lookup], subagents=[researcher])

## 단계 2

한 가지 동작을 실행하고 출력으로 확인합니다.

In [4]:
# 부모가 source-reviewer에게 release 확인을 위임합니다.
result = agent.invoke({'messages':[{'role':'user','content':"task 도구로 source-reviewer에게 source_lookup('release')를 실행해 주장을 확인하게 하고 결과를 한국어로 요약하세요."}]})
calls = [call for m in result['messages'] for call in getattr(m, 'tool_calls', [])]
tasks = [m for m in result['messages'] if type(m).__name__ == 'ToolMessage' and m.name == 'task']
print('부모 호출:', [call['name'] for call in calls])
print('위임 반환:', str(tasks[0].content)[:500])
assert any(call['name'] == 'task' for call in calls)
assert tasks and 'planning opt-in' in str(tasks[0].content).lower()

부모 호출: ['task']
위임 반환: (1) Verified claims:
- The source supports the claim that “DeepAgents 0.7 makes planning opt-in.”

(2) Unsupported or contradicted claims:
- No unsupported or contradicted claims are present in the provided source text.

(3) Confidence assessment:
- High confidence. The source is a single, direct statement that clearly supports the claim and contains no conflicting information.
